<a href="https://colab.research.google.com/github/Mike-Mans/Math156-Final-Project/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install optuna

import os
import torch
import torchvision
import torchaudio
import numpy as np
import pandas as pd
import sklearn
import librosa
import optuna
from optuna.storages import JournalStorage, JournalFileStorage
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
import glob
import torch.nn as nn
import random
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.cuda.amp import GradScaler, autocast

def set_seed(seed_value=42):
    """sets seed for reproducibility"""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        # ensure deterministic behavior for cuDNN
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

DRIVE_SAVE_DIR = '/kaggle/working/Math156Project'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"checkpoints and reports saved to: {DRIVE_SAVE_DIR}")


# target class definitions
CORE_COMMANDS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
UNKNOWN_WORDS = ['bed', 'cat', 'dog', 'eight', 'five', 'four', 'happy', 'house', 'marvin', 'nine',
                 'one', 'seven', 'sheila', 'six', 'three', 'tree', 'two', 'wow', 'zero']

def get_label(word_class):
    """maps word class to a numeric label (12-class)."""
    if word_class in CORE_COMMANDS:
        return CORE_COMMANDS.index(word_class)
    elif word_class in UNKNOWN_WORDS:
        return 10  # unknown
    elif word_class == '_silence_':
        return 11  # silence
    else:
        return 10  # default to unknown

def create_silence_data():
    """creates 1-second silence chunks from background noise files."""

    silence_dir = os.path.join(DATASET_PATH, "SpeechCommands", "speech_commands_v0.02", "_silence_")
    background_noise_dir = os.path.join(DATASET_PATH, "SpeechCommands", "speech_commands_v0.02", "_background_noise_")

    # ensure silence directory exists
    os.makedirs(silence_dir, exist_ok=True)

    existing_files = glob.glob(os.path.join(silence_dir, "*.wav"))
    if len(existing_files) > 0:
        print(f"found {len(existing_files)} existing silence files. skipping generation.")
        return existing_files

    silence_files = []
    background_files = glob.glob(os.path.join(background_noise_dir, "*.wav"))
    chunk_counter = 0

    for bg_file in background_files:
        waveform, sample_rate = torchaudio.load(bg_file)
        assert sample_rate == 16000, f"expected 16khz, got {sample_rate}"

        # use first channel if multi-channel
        if waveform.shape[0] > 1:
            waveform = waveform[0:1, :]

        waveform = waveform.squeeze()
        total_samples = waveform.shape[0]
        chunk_size = 16000
        num_chunks = total_samples // chunk_size

        # slice into 1s chunks
        for i in range(num_chunks):
            start_sample = i * chunk_size
            end_sample = start_sample + chunk_size
            chunk = waveform[start_sample:end_sample]

            # save chunk
            chunk_filename = f"silence_{chunk_counter:05d}.wav"
            chunk_path = os.path.join(silence_dir, chunk_filename)
            torchaudio.save(chunk_path, chunk.unsqueeze(0), sample_rate)

            silence_files.append(chunk_path)
            chunk_counter += 1

    print(f"created {len(silence_files)} silence chunks from {len(background_files)} background noise files")
    return silence_files

def load_dataset_splits():
    """
    loads validation/testing lists and creates master file lists.
    returns train_files, val_files, test_files, background_noise_files
    """
    dataset_dir = os.path.join(DATASET_PATH, "SpeechCommands", "speech_commands_v0.02")

    with open(os.path.join(dataset_dir, 'validation_list.txt'), 'r') as f:
        val_lines = f.read().splitlines()
    with open(os.path.join(dataset_dir, 'testing_list.txt'), 'r') as f:
        test_lines = f.read().splitlines()

    val_files_set = set(val_lines)
    test_files_set = set(test_lines)

    # find all wav files
    all_wav_files = []
    for root, dirs, files in os.walk(dataset_dir):
        for file in files:
            if file.endswith('.wav'):
                rel_path = os.path.relpath(os.path.join(root, file), dataset_dir)
                all_wav_files.append(rel_path)

    train_files, val_files, test_files = [], [], []

    for wav_file in all_wav_files:
        if wav_file in val_files_set:
            val_files.append(os.path.join(dataset_dir, wav_file))
        elif wav_file in test_files_set:
            test_files.append(os.path.join(dataset_dir, wav_file))
        else:
            # exclude background and silence noise
            if "_background_noise_" not in wav_file and "_silence_" not in wav_file:
                 train_files.append(os.path.join(dataset_dir, wav_file))


    # partition silence files (80/10/10)
    silence_dir = os.path.join(DATASET_PATH, "SpeechCommands", "speech_commands_v0.02", "_silence_")
    if os.path.exists(silence_dir):
        silence_files = glob.glob(os.path.join(silence_dir, "*.wav"))
        random.Random(SEED).shuffle(silence_files)

        n_silence = len(silence_files)
        n_train_silence = int(0.8 * n_silence)
        n_val_silence = int(0.1 * n_silence)

        train_files.extend(silence_files[:n_train_silence])
        val_files.extend(silence_files[n_train_silence:n_train_silence + n_val_silence])
        test_files.extend(silence_files[n_train_silence + n_val_silence:])
    else:
        print(f"warning: silence directory not found at {silence_dir}.")

    # get background noise files
    background_noise_dir = os.path.join(dataset_dir, "_background_noise_")
    background_files = [f for f in glob.glob(os.path.join(background_noise_dir, "*.wav"))]

    print(f"dataset splits created:")
    print(f"  training: {len(train_files)} files")
    print(f"  validation: {len(val_files)} files")
    print(f"  testing: {len(test_files)} files")
    print(f"  background noise files: {len(background_files)}")

    return train_files, val_files, test_files, background_files

DATASET_PATH = "./speech_commands_v2"
os.makedirs(DATASET_PATH, exist_ok=True)

# download dataset
torchaudio.datasets.SPEECHCOMMANDS(
    root=DATASET_PATH, folder_in_archive="SpeechCommands", download=True, subset=None
)
print("download complete")

# create silence chunks and load splits
create_silence_data()
train_files, val_files, test_files, bg_noise_paths = load_dataset_splits()

print("pre-loading background noise files")
bg_noise_waveforms = []
for p in bg_noise_paths:
    try:
        wf, sr = torchaudio.load(p)
        if sr != 16000:
            wf = torchaudio.functional.resample(wf, sr, 16000)
        bg_noise_waveforms.append(wf.squeeze()) # remove channel dim
    except Exception as e:
        print(f"warning: could not load bg noise {p}: {e}")
print(f"loaded {len(bg_noise_waveforms)} noise files.")

class SpeechCommandsDataset(Dataset):
    """custom dataset for loading, processing, and augmenting speech data."""
    def __init__(
        self,
        file_paths,
        bg_noise_waveforms_list=None,
        mode="train",
        sample_rate=16000,
        n_fft=400,
        hop_length=160,
        time_mask_param=20,
        freq_mask_param=10,
    ):
        self.file_paths = file_paths
        self.mode = mode
        self.sample_rate = sample_rate
        self.num_samples = sample_rate  # 1 second

        # spectrogram transform
        self.mel_spectrogram_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=128,
            power=2
        )

        # training augmentations
        self.bg_noise_waveforms = []
        if mode == "train":
            self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=time_mask_param)
            self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=freq_mask_param)

            # load background noise
            if bg_noise_waveforms_list:
                self.bg_noise_waveforms = bg_noise_waveforms_list
        else:
            self.time_mask = None
            self.freq_mask = None

    def __len__(self):
        return len(self.file_paths)

    def _load_and_fix_waveform(self, wav_path):
        """loads, resamples, converts to mono, and pads/truncates to 1 second."""
        try:
            waveform, sr = torchaudio.load(wav_path)
        except Exception as e:
            print(f"error loading {wav_path}: {e}")
            return torch.zeros((1, self.num_samples)), False # return silence on error

        if sr != self.sample_rate:
            waveform = torchaudio.functional.resample(waveform, sr, self.sample_rate)

        # ensure mono
        if waveform.shape[0] > 1:
            waveform = waveform[:1, :]

        # pad or truncate
        num_current = waveform.shape[1]
        if num_current < self.num_samples:
            pad_amount = self.num_samples - num_current
            waveform = F.pad(waveform, (0, pad_amount))
        elif num_current > self.num_samples:
            waveform = waveform[:, : self.num_samples]

        return waveform, True

    def _get_label_from_path(self, wav_path):
        """extracts class name from path and converts to label index."""
        word_class = os.path.basename(os.path.dirname(wav_path))
        return get_label(word_class)

    def _random_time_shift(self, waveform, max_shift_seconds=0.2):
        """randomly shifts waveform left/right."""
        max_shift = int(max_shift_seconds * self.sample_rate)
        if max_shift == 0:
            return waveform

        shift = random.randint(-max_shift, max_shift)
        if shift > 0: # shift right
            waveform = F.pad(waveform, (shift, 0))[:, :-shift]
        elif shift < 0: # shift left
            shift = -shift
            waveform = F.pad(waveform, (0, shift))[:, shift:]
        return waveform

    def _add_random_noise(self, waveform, min_snr=5, max_snr=20):
        """adds a random snippet of background noise."""
        if not self.bg_noise_waveforms:
            return waveform

        # pick random noise/snippet
        noise_wf = random.choice(self.bg_noise_waveforms)
        if noise_wf.shape[0] > self.num_samples:
            start_idx = random.randint(0, noise_wf.shape[0] - self.num_samples)
            noise_snippet = noise_wf[start_idx : start_idx + self.num_samples]
        else:
            noise_snippet = noise_wf # fallback if noise is short
            pad_amount = self.num_samples - noise_snippet.shape[0]
            noise_snippet = F.pad(noise_snippet, (0, pad_amount))

        # calculate power
        speech_power = waveform.norm(p=2)
        noise_power = noise_snippet.norm(p=2)

        # pick snr and scale
        snr_db = random.uniform(min_snr, max_snr)
        snr_lin = 10.0 ** (snr_db / 10.0)
        scale_factor = (speech_power / (noise_power * (snr_lin**0.5) + 1e-9))

        # mix and clamp
        noisy_waveform = (waveform + noise_snippet * scale_factor).clamp(-1.0, 1.0)
        return noisy_waveform

    def _waveform_to_log_spectrogram(self, waveform):
        """converts waveform to a log-scaled, instance-normalized spectrogram."""
        spectrogram = self.mel_spectrogram_transform(waveform)
        log_spectrogram = torch.log(spectrogram + 1e-8)

        return log_spectrogram

    def __getitem__(self, idx):
        wav_path = self.file_paths[idx]

        # 1. load
        waveform, load_success = self._load_and_fix_waveform(wav_path)

        # 2. waveform augmentations (train)
        if self.mode == "train":
            waveform = self._random_time_shift(waveform)
            # handle channel dim for mixing
            waveform = self._add_random_noise(waveform.squeeze(0)).unsqueeze(0)

        # 3. to spectrogram
        log_spec = self._waveform_to_log_spectrogram(waveform)

        # 4. spec augmentations (train)
        if self.mode == "train" and self.time_mask is not None:
             log_spec = self.time_mask(log_spec)
             log_spec = self.freq_mask(log_spec)

        # 5. label
        if load_success:
            label = self._get_label_from_path(wav_path)
        else:
            label = 11  # 11 is the label for '_silence_'

        label = torch.tensor(label, dtype=torch.long)

        return log_spec, label

class ShallowSpeechCNN(nn.Module):
    """a shallow cnn for speech classification."""
    def __init__(self, num_classes):
        super(ShallowSpeechCNN, self).__init__()
        # conv block 1
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # (batch, 16, h/2, w/2)

        # conv block 2
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # (batch, 32, h/4, w/4)

        # adaptive pooling
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        # (batch, 32, 4, 4)

        # fully connected
        self.fc1 = nn.Linear(32 * 4 * 4, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        # conv block 1
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        # conv block 2
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        # adaptive pooling
        x = self.adaptive_pool(x)
        # flatten
        x = torch.flatten(x, start_dim=1)
        # fc layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x) # logits
        return x

def train_epoch(model, data_loader, criterion, optimizer, device, scaler):
    """runs a single training epoch."""
    model.train()
    total_loss = 0
    total_samples = 0
    # progress bar
    for inputs, labels in tqdm.tqdm(data_loader, desc="training epoch", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        total_samples += labels.size(0)

        # forward pass with autocast
        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        # backward pass with scaler
        optimizer.zero_grad()
        scaler.scale(loss).backward() # use scaler to scale loss
        scaler.step(optimizer)       # use scaler to step
        scaler.update()              # update scaler

        total_loss += loss.item() * inputs.size(0)
    if total_samples == 0: return 0
    return total_loss / total_samples

def val_epoch(model, data_loader, criterion, device):
    """runs a single validation epoch."""
    model.eval()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0
    with torch.no_grad():
        for inputs, labels in tqdm.tqdm(data_loader, desc="validation epoch", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)

            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    if total_samples == 0: return 0, 0
    avg_loss = total_loss / total_samples
    accuracy = correct_predictions / total_samples
    return avg_loss, accuracy

# optuna runs separate small trials

optuna_train_labels = [get_label(os.path.basename(os.path.dirname(f))) for f in train_files]

optuna_train_files_subset, _ = train_test_split(
    train_files,
    train_size=0.25,
    shuffle=True,
    random_state=42,
    stratify=optuna_train_labels
)

def objective(trial):
    """optuna objective function to maximize validation accuracy."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = 12
    num_optuna_epochs = 3 # fewer epochs for speed
    num_workers = 2

    # 1. suggest hyperparameters
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    time_mask_param = trial.suggest_int("time_mask_param", 10, 40)
    freq_mask_param = trial.suggest_int("freq_mask_param", 5, 20)

    # 2. create dataloaders
    train_dataset_optuna = SpeechCommandsDataset(
        optuna_train_files_subset,
        bg_noise_waveforms_list=bg_noise_waveforms,
        mode="train",
        time_mask_param=time_mask_param,
        freq_mask_param=freq_mask_param
    )
    val_dataset_optuna = SpeechCommandsDataset(val_files, mode="val")

    train_loader_optuna = DataLoader(
        train_dataset_optuna, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader_optuna = DataLoader(
        val_dataset_optuna, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    # 3. create model and optimizer
    model_optuna = ShallowSpeechCNN(num_classes=num_classes).to(device)
    criterion_optuna = nn.CrossEntropyLoss()
    scaler_optuna = GradScaler()

    if optimizer_name == "Adam":
        optimizer_optuna = torch.optim.Adam(model_optuna.parameters(), lr=lr)
    elif optimizer_name == "RMSprop":
        optimizer_optuna = torch.optim.RMSprop(model_optuna.parameters(), lr=lr)
    else: # sgd
        optimizer_optuna = torch.optim.SGD(model_optuna.parameters(), lr=lr, momentum=0.9)

    # 4. training and validation loop
    best_val_accuracy = 0.0
    for epoch in range(num_optuna_epochs):
        print(f"  trial {trial.number}, epoch {epoch+1}/{num_optuna_epochs}")
        # reuse train/val functions
        train_loss = train_epoch(model_optuna, train_loader_optuna, criterion_optuna, optimizer_optuna, device, scaler_optuna)
        val_loss, val_accuracy = val_epoch(model_optuna, val_loader_optuna, criterion_optuna, device)
        best_val_accuracy = max(best_val_accuracy, val_accuracy)

        trial.report(val_accuracy, epoch)
        if trial.should_prune():
            print(f"  trial {trial.number} pruned.")
            raise optuna.TrialPruned()

    return best_val_accuracy

optuna_study_path = os.path.join(DRIVE_SAVE_DIR, "optuna_study.log")
study_name = "speech_cnn_v1"

storage = JournalStorage(JournalFileStorage(optuna_study_path))

# create or load study
try:
    study = optuna.load_study(study_name=study_name, storage=storage)
    print(f"loaded existing Optuna study '{study_name}'.")
    print(f"  study already has {len(study.trials)} trials.")
except KeyError:
    study = optuna.create_study(study_name=study_name, storage=storage, direction="maximize")
    print(f"created new optuna study '{study_name}'.")

# run optimization
try:
    n_target_total_trials = 10

    # calculate how many trials left to run
    n_existing_trials = len(study.trials)
    n_new_trials = n_target_total_trials - n_existing_trials

    if n_new_trials > 0:
        print(f"study has {n_existing_trials} trials. running {n_new_trials} new trials")
        study.optimize(objective, n_trials=n_new_trials)
    else:
        print(f"Study has already completed {n_existing_trials} trials (target was {n_target_total_trials}).")
except KeyboardInterrupt:
    print("optuna optimization interrupted. study progress is saved.")
    pass

print("optuna search complete. using best parameters for final training.")
best_params = study.best_trial.params

LEARNING_RATE = best_params['lr']
BATCH_SIZE = best_params['batch_size']
OPTIMIZER_NAME = best_params['optimizer']
TIME_MASK = best_params['time_mask_param']
FREQ_MASK = best_params['freq_mask_param']

print(f"best params: {best_params}")

# setup
num_classes = 12
num_epochs = 20
batch_size = BATCH_SIZE
num_workers = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = os.path.join(DRIVE_SAVE_DIR, 'my_cnn_checkpoint.pth')


# create datasets and dataloaders
train_dataset = SpeechCommandsDataset(train_files, bg_noise_waveforms_list=bg_noise_waveforms, mode="train", time_mask_param=TIME_MASK, freq_mask_param=FREQ_MASK)
val_dataset = SpeechCommandsDataset(val_files, mode="val")
test_dataset = SpeechCommandsDataset(test_files, mode="test")

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

# model setup
model = ShallowSpeechCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()

if OPTIMIZER_NAME == "Adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
elif OPTIMIZER_NAME == "RMSprop":
    optimizer = torch.optim.RMSprop(model.parameters(), lr=LEARNING_RATE)
else: # sgd
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)

scaler = GradScaler()




# history for current run
train_losses, val_losses, val_accuracies = [], [], []
best_val_loss = float('inf')
start_epoch = 0

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if 'scaler_state_dict' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1  # start from next epoch
    best_val_loss = checkpoint['best_val_loss']

    # load history to continue plotting correctly
    train_losses = checkpoint.get('train_losses_history', [])
    val_losses = checkpoint.get('val_losses_history', [])
    val_accuracies = checkpoint.get('val_accuracies_history', [])

    print(f"resuming training from epoch {start_epoch}")
else:
    print("no checkpoint found, starting training from scratch")


# TRAINING LOOP
print(f"starting training for {num_epochs} epochs")
print(f"device: {device}")
print(f"best checkpoint saved to: {checkpoint_path}")

for epoch in range(start_epoch, num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss, val_accuracy = val_epoch(model, val_loader, criterion, device)

    # append results to history
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f"epoch {epoch+1}: train loss: {train_loss:.4f}, val loss: {val_loss:.4f}, val acc: {val_accuracy:.4f}", flush=True)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        print(f"new best model found, saving checkpoint to {checkpoint_path}...")

        # save history with model state
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_loss': best_val_loss,
            'train_losses_history': train_losses,
            'val_losses_history': val_losses,
            'val_accuracies_history': val_accuracies
        }
        torch.save(checkpoint, checkpoint_path)
    else:
        print(f"validation loss did not improve. current best: {best_val_loss:.4f}")



# path for report plot
cm_plot_path = os.path.join(DRIVE_SAVE_DIR, 'confusion_matrix.png')


# load best model from main run
if os.path.exists(checkpoint_path):
    print(f"\nloading best model from main training run: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # re-instantiate model
    model = ShallowSpeechCNN(num_classes=num_classes).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    print(f"model loaded. trained for {checkpoint.get('epoch', 'n/a')+1} epochs.")
    print(f"best val loss was: {checkpoint.get('best_val_loss', 'n/a'):.4f}")

    # check for saved history
    if 'train_losses_history' in checkpoint:
        train_hist = checkpoint['train_losses_history']
        val_hist = checkpoint['val_losses_history']
        acc_hist = checkpoint['val_accuracies_history']
        epochs_range = range(1, len(train_hist) + 1)

        # 1. plot loss
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(epochs_range, train_hist, 'bo-', label='Training Loss')
        plt.plot(epochs_range, val_hist, 'ro-', label='Validation Loss')
        plt.title('Training and Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()

        # 2. plot accuracy
        plt.subplot(1, 2, 2)
        plt.plot(epochs_range, acc_hist, 'go-', label='Validation Accuracy')
        plt.title('Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()

        plt.suptitle(f'Training Performance (Epoch {checkpoint.get("epoch", "n/a")+1})')
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])

        # save combined plot
        combined_plot_path = os.path.join(DRIVE_SAVE_DIR, 'training_performance_plots.png')
        plt.savefig(combined_plot_path)
        print(f"performance plots saved to {combined_plot_path}")
        plt.show()
    else:
        print("warning: training history not found in checkpoint file.")

    # run test
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm.tqdm(test_loader, desc="testing"):
            inputs, labels = inputs.to(device), labels.to(device)

            with autocast():
                outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("evaluation complete.")

    # classification report
    class_names = [
        'yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go',
        '_unknown_', '_silence_'
    ]

    report = classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)
    print(report)

    # confusion matrix
    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix (Test Set)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()

    # save confusion matrix plot
    plt.savefig(cm_plot_path)
    plt.show()

else:
    print(f"\ncheckpoint file not found at {checkpoint_path}.")
    print("training did not complete or save a model. skipping final evaluation.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 7.5 MB/s eta 0:00:00
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/chec

KeyboardInterrupt: 